# Strands Agents with AgentCore Memory (Long term memory)

## Introduction

This tutorial demonstrates how to build an **intelligent customer support agent** using Strands agents integrated with AgentCore Memory via hooks. We'll focus on Long term memory for customer interaction history, remembering purchase details, and provides personalized support based on previous conversations and user preferences.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long term Conversational                                                         |
| Agent type          | Customer Support                                                                 |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| Tutorial components | AgentCore Semantic and User Preferences Memory Extraction, Hooks for storing and retrieving Memory              |
| Example complexity  | Intermediate                                                                     |

You'll learn to:
- Set up AgentCore Memory with Long term strategies
- Create memory hooks for automatic storage and retrieval
- Build a customer support agent with persistent memory
- Handle customer issues with context from previous interactions

### Scenario Context
In this example, we will build a **Customer Support Use Case**. The agent will remember customer context, including order history, preferences, and previous issues, enabling more personalized and effective support. Conversations with customers are automatically stored using memory hooks, ensuring that important details are never lost. By employing multiple memory strategies such as semantic, and User Preference — the agent can capture a wide range of relevant information. This setup allows the agent to resolve issues with full awareness of the customer's history and preferences. Additionally, the agent is integrated with web search capabilities, making it easy to provide up-to-date product information and troubleshooting guidance as needed.

## Architecture

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## Prerequisites

To execute this tutorial you will need:
- Python 3.10+
- AWS credentials with Amazon Bedrock AgentCore Memory permissions
- Amazon Bedrock AgentCore SDK

## Step 1: Install Dependencies and Setup
Let's begin importing all the necessary libraries and defining the clients to make this notebook work.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
import json
from typing import Dict, List
from datetime import datetime
from botocore.exceptions import ClientError

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("customer-support")

# Import required modules
from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from ddgs import DDGS

# Import memory management modules
from bedrock_agentcore.memory import MemoryClient, MemorySessionManager
from bedrock_agentcore.memory.constants import (
    StrategyType, ConversationalMessage, MessageRole, RetrievalConfig
)
from bedrock_agentcore.memory.models import (
    StringValue, EventMetadataFilter, LeftExpression, RightExpression, OperatorType
)

# Define message role constants
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT


In [ ]:
# Configuration - Replace with the correct values
REGION = "us-west-2"
CUSTOMER_ID = "customer_001"
SESSION_ID = f"support_{datetime.now().strftime('%Y%m%d%H%M%S')}"

logger.info(f"✅ Configuration loaded")
logger.info(f"   Region: {REGION}")
logger.info(f"   Customer ID: {CUSTOMER_ID}")
logger.info(f"   Session ID: {SESSION_ID}")

## Step 2: Create Memory Resource for Customer Support

For customer support, we'll use multiple memory strategies:
- **USER_PREFERENCE**: Captures customer preferences and behavior
- **SEMANTIC**: Stores order facts and product information

In [ ]:
# Initialize Memory Client
client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

# Define memory strategies for customer support
strategies = [
    {
        StrategyType.USER_PREFERENCE.value: {
            "name": "CustomerPreferences",
            "description": "Captures customer preferences and behavior",
            "namespaces": ["support/customer/{actorId}/preferences/"]
        }
    },
    {
        StrategyType.SEMANTIC.value: {
            "name": "CustomerSupportSemantic",
            "description": "Stores facts from conversations",
            "namespaces": ["support/customer/{actorId}/semantic/"],
        }
    }
]

# Create memory resource
try:
    memory = client.create_memory_and_wait(
        name=memory_name,
        strategies=strategies,         # Define the memory strategies
        description="Memory for customer support agent",
        event_expiry_days=90,          # Memories expire after 90 days
    )
    memory_id = memory['id']
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    logger.error(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if 'memory_id' in locals():
        try:
            client.delete_memory_and_wait(memoryId=memory_id, max_wait=300)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")

Let's confirm if our memory contains the strategies we assigned

In [ ]:
strategies = client.get_memory_strategies(memory_id)
print(json.dumps(strategies, indent=2, default=str))

## Step 3: Create Agent Tools

In [ ]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

@tool
def web_search(query: str, max_results: int = 3) -> str:
    """Search the web for product information, troubleshooting guides, or support articles.
    
    Args:
        query: Search query for product info or troubleshooting
        max_results: Maximum number of results to return
    
    Returns:
        Search results with titles and snippets
    """
    try:
        results = DDGS().text(query, region="us-en", max_results=max_results)
        if not results:
            return "No search results found."
        
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(f"{i}. {result.get('title', 'No title')}\n   {result.get('body', 'No description')}")
        
        return "\n".join(formatted_results)
    except RatelimitException:
        return "Rate limit reached: Please try again after a short delay."
    except DuckDuckGoSearchException as d:
        return f"Search Error: {d}"
    except Exception as e:
        return f"Search error: {str(e)}"

logger.info("✅ Web search tool ready")

@tool
def check_order_status(order_number: str) -> str:
    """Check the status of a customer order.
    
    Args:
        order_number: The order number to check
    
    Returns:
        Order status information
    """
    # Simulate order lookup
    mock_orders = {
        "123456": "iPhone 15 Pro - Delivered on June 5, 2025",
        "654321": "Sennheiser Headphones - Delivered on June 25, 2025, 1-year warranty active",
        "789012": "Samsung Galaxy S23 - In transit, expected delivery on July 1, 2025",
    }
    
    return mock_orders.get(order_number, f"Order {order_number} not found. Please verify the order number.")

logger.info("✅ Check Order Status tool ready")

## Step 4: Initialize Session Manager

Now we'll create a MemorySessionManager and MemorySession for our customer. The session manager provides a cleaner API by automatically handling memory_id, actor_id, and session_id parameters in all operations.

In [ ]:
# Initialize the session manager
session_manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)

# Create a memory session for the specific customer
customer_session = session_manager.create_memory_session(
    actor_id=CUSTOMER_ID,
    session_id=SESSION_ID
)

logger.info(f"✅ Session manager initialized for memory: {memory_id}")
logger.info(f"✅ Customer session created for actor: {CUSTOMER_ID}")
logger.info(f"   Session ID: {SESSION_ID}")

## Step 5: Create Memory Hook Provider for Customer Support
Hooks are special functions that run at specific points in an agent's execution lifecycle. Our custom hook provider will automatically manage customer support context by:
- **Saving support interactions** after each response using `add_turns()` with ConversationalMessage objects
- **Retrieving and Injecting relevant context** from previous orders and preferences when processing new queries using `search_long_term_memories()`

This creates a seamless memory experience without manual management, and the session-based API eliminates the need to pass memory_id, actor_id, and session_id repeatedly.

In [ ]:
# Helper function to get namespaces from memory strategies list
def get_namespaces(mem_client: MemoryClient, memory_id: str) -> Dict:
    """Get namespace mapping for memory strategies."""
    strategies = mem_client.get_memory_strategies(memory_id)
    return {i["type"]: i["namespaces"][0] for i in strategies}

# Get namespaces for retrieval configuration
namespaces_dict = get_namespaces(client, memory_id)
logger.info(f"✅ Retrieved {len(namespaces_dict)} namespace mappings")
for strategy_type, namespace in namespaces_dict.items():
    logger.info(f"   {strategy_type}: {namespace}")

In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    """Memory hooks for customer support agent using MemorySession"""
    
    def __init__(self, customer_session, namespaces: Dict):
        """Initialize with a MemorySession instance and namespace mappings
        
        Args:
            customer_session: MemorySession instance for the customer
            namespaces: Dictionary mapping strategy types to namespace templates
        """
        self.customer_session = customer_session
        self.namespaces = namespaces
        
        # Define retrieval configuration for different memory types
        self.retrieval_config = {
            "userPreferenceMemoryStrategy": RetrievalConfig(top_k=3, relevance_score=0.5),
            "semanticMemoryStrategy": RetrievalConfig(top_k=5, relevance_score=0.3)
        }
    
    def retrieve_customer_context(self, event: MessageAddedEvent):
        """Retrieve customer context before processing support query using MemorySession"""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                # Use MemorySession for context retrieval
                relevant_memories = []
                
                # Search across different memory namespaces using MemorySession
                for context_type, namespace_template in self.namespaces.items():
                    # Resolve namespace template with actual actor ID from session
                    resolved_namespace = namespace_template.format(
                        actorId=self.customer_session._actor_id
                    )
                    
                    # Get retrieval config for this strategy type
                    config = self.retrieval_config.get(
                        context_type,
                        RetrievalConfig(top_k=3, relevance_score=0.3)
                    )
                    
                    # Use MemorySession API (no need to pass actor_id/session_id)
                    memories = self.customer_session.search_long_term_memories(
                        query=user_query,
                        namespace_prefix=resolved_namespace,
                        top_k=config.top_k
                    )
                    
                    # Filter by relevance score
                    filtered_memories = [
                        memory for memory in memories
                        if memory.get("score", 0) >= config.relevance_score
                    ]
                    
                    relevant_memories.extend(filtered_memories)
                    logger.info(f"Found {len(filtered_memories)} relevant memories in {context_type} (filtered from {len(memories)} total)")
                
                # Inject context into agent's system prompt if memories found
                if relevant_memories:
                    context_text = self._format_context(relevant_memories)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = (
                        f"Customer Context:\n{context_text}\n\n{original_text}"
                    )
                    logger.info(f"✅ Injected {len(relevant_memories)} memories into agent context")
                    
            except Exception as e:
                logger.error(f"Failed to retrieve customer context: {e}")
    
    def _format_context(self, memories: List[Dict]) -> str:
        """Format retrieved memories for agent context"""
        context_lines = []
        for i, memory in enumerate(memories[:5], 1):  # Limit to top 5
            content = memory.get('content', {}).get('text', 'No content available')
            score = memory.get('score', 0)
            context_lines.append(f"{i}. (Score: {score:.2f}) {content[:200]}...")
        
        return "\n".join(context_lines)
    
    def save_support_interaction(self, event: AfterInvocationEvent):
        """Save support interaction using MemorySession (cleaner API)"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last customer query and agent response
                customer_query = None
                agent_response = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                if customer_query and agent_response:
                    # Use MemorySession with ConversationalMessage objects
                    interaction_messages = [
                        ConversationalMessage(customer_query, USER),
                        ConversationalMessage(agent_response, ASSISTANT)
                    ]
                    
                    result = self.customer_session.add_turns(interaction_messages)
                    logger.info(f"✅ Saved interaction using MemorySession - Event ID: {result['eventId']}")
            
        except Exception as e:
            logger.error(f"Failed to save support interaction: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """Register customer support memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)
        logger.info("✅ Customer support memory hooks registered with MemorySession support")

### Step 6: Create Customer Support Agent

In [ ]:
# Create memory hooks for customer support with MemorySession
support_hooks = CustomerSupportMemoryHooks(customer_session, namespaces_dict)

# Create customer support agent
support_agent = Agent(
    hooks=[support_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[web_search, check_order_status],
    system_prompt="""You are a helpful customer support agent with access to customer history and order information. 
    
    Your role:
    - Help customers with their orders, returns, and product issues
    - Use customer context to provide personalized support
    - Search for product information when needed
    - Be empathetic and solution-focused
    - Reference previous orders and preferences when relevant
    
    Always be professional, helpful, and aim to resolve customer issues efficiently."""
)

logger.info("✅ Customer support agent created with MemorySession integration")
logger.info(f"   Customer: {CUSTOMER_ID}")
logger.info(f"   Session: {SESSION_ID}")

### Step 7: Seed Customer History

Let's add some previous customer interactions to demonstrate memory functionality.

**NOTE: This section uses ConversationalMessage format and session-based storage.**

In [ ]:
# Seed with previous customer interactions using MemorySession
previous_interactions = [
    ConversationalMessage("I bought a new iPhone 15 Pro on June 1st, 2025. Order number is 123456.", USER),
    ConversationalMessage("Thank you for your purchase! I can see your iPhone 15 Pro order #123456 was delivered successfully. How can I help you today?", ASSISTANT),
    ConversationalMessage("I also ordered Sennheiser headphones on June 20th. Order number 654321. They came with 1-year warranty.", USER),
    ConversationalMessage("Perfect! I have your Sennheiser headphones order #654321 on file with the 1-year warranty. Both your iPhone and headphones should work great together.", ASSISTANT),
    ConversationalMessage("I'm looking for a good laptop. I prefer ThinkPad models.", USER),
    ConversationalMessage("Great choice! ThinkPads are excellent for their durability and performance. Let me help you find the right model for your needs.", ASSISTANT)
]

# Save using MemorySession
try:
    event_response = customer_session.add_turns(previous_interactions)
    logger.info(f"✅ Seeded customer history using MemorySession")
    logger.info(f"   Event ID: {event_response['eventId']}")
except Exception as e:
    logger.error(f"⚠️ Error seeding history: {e}")

#### Agent is ready to go. 

### Lets test Customer Support Scenarios

In [ ]:
# Test 1: Customer reports iPhone issue
response1 = support_agent("My iPhone is running very slow and gets hot when charging. Can you help?")
print(f"Support Agent: {response1}")

In [ ]:
# Test 2: Bluetooth connectivity issue
response2 = support_agent("My iPhone won't connect to my Sennheiser headphones via Bluetooth. How do I fix this?")
print(f"Support Agent: {response2}")

In [ ]:
# Test 3: Check order status
response3 = support_agent("Can you check the status of my recent orders?")
print(f"Support Agent: {response3}")

In [ ]:
# Test 4: Product recommendation based on preferences
response4 = support_agent("I'm still interested in buying a laptop. What ThinkPad models do you recommend?")
print(f"Support Agent: {response4}")

### Verify Customer Memory Storage

In [ ]:
# Check stored customer memories using MemorySession
print("\n📚 Customer Memory Summary:")
print("=" * 50)

for context_type, namespace_template in namespaces_dict.items():
    namespace = namespace_template.replace("{actorId}", CUSTOMER_ID)
    
    try:
        memories = customer_session.search_long_term_memories(
            query="customer orders and preferences",
            namespace_prefix=namespace,
            top_k=3
        )
        
        print(f"\n{context_type.upper()} ({len(memories)} items):")
        for i, memory in enumerate(memories, 1):
            if isinstance(memory, dict):
                content = memory.get('content', {})
                score = memory.get('score', 0)
                if isinstance(content, dict):
                    text = content.get('text', '')[:150] + "..."
                    print(f"  {i}. [Score: {score:.2f}] {text}")
                    
    except Exception as e:
        logger.error(f"Error retrieving {context_type} memories: {e}")

print("\n" + "=" * 50)

## Advanced Features: Branching and Metadata

### Conversation Branching for Support Scenarios

Branching allows exploring alternative support paths. Useful for:
- Testing different resolution approaches
- Escalation scenarios
- Alternative product recommendations

In [ ]:
# Get the last event ID from our conversation
events = customer_session.list_events()
if events:
    last_event_id = events[-1].event_id
    
    # Fork conversation to explore escalation path
    branch_event = customer_session.fork_conversation(
        root_event_id=last_event_id,
        branch_name="escalation-path",
        messages=[
            ConversationalMessage(
                "I'm not satisfied with this solution. I want to speak to a manager.",
                USER
            ),
            ConversationalMessage(
                "I understand your frustration. Let me escalate this to our senior support team. They'll contact you within 2 hours with a priority resolution.",
                ASSISTANT
            )
        ]
    )
    
    logger.info(f"✅ Created escalation branch from event {last_event_id}")
    
    # List all branches
    branches = customer_session.list_branches()
    print(f"\n🌳 Support session has {len(branches)} branch(es):")
    for branch in branches:
        print(f"   - {branch.name}: {branch.event_count} events")
else:
    print("No events found to branch from")

### Metadata for Support Analytics

Track support metrics with metadata:
- Issue severity and category
- Resolution status
- Customer satisfaction
- Response time tracking

In [ ]:
from bedrock_agentcore.memory.models import StringValue

# Add a support interaction with rich metadata
metadata_event = customer_session.add_turns(
    messages=[
        ConversationalMessage(
            "The replacement arrived and works perfectly! Thank you for the quick resolution.",
            USER
        ),
        ConversationalMessage(
            "Wonderful! I'm glad we could resolve this quickly. Is there anything else I can help you with today?",
            ASSISTANT
        )
    ],
    metadata={
        "issue_type": StringValue("defective_product"),
        "severity": StringValue("medium"),
        "resolution_status": StringValue("resolved"),
        "customer_satisfaction": StringValue("satisfied"),
        "resolution_time_hours": StringValue("24"),
        "required_escalation": StringValue("false"),
        "product_category": StringValue("electronics")
    }
)

logger.info(f"✅ Added support event with metadata - Event ID: {metadata_event['eventId']}")
print("\n📊 Support event tagged with:")
print("   - Issue Type: defective_product")
print("   - Severity: medium")
print("   - Resolution Status: resolved")
print("   - Customer Satisfaction: satisfied")
print("   - Resolution Time: 24 hours")

### Querying Support Metrics

Analyze support performance using metadata filters:

In [ ]:
from bedrock_agentcore.memory.models import EventMetadataFilter, LeftExpression, RightExpression, OperatorType

try:
    # Query resolved issues
    resolved_events = customer_session.list_events(
        metadata_filter=EventMetadataFilter(
            expression="resolution_status = :status",
            values={
                ":status": StringValue("resolved")
            }
        )
    )
    
    print(f"\n✅ Found {len(resolved_events)} resolved issue(s)")
    
    # Query satisfied customers
    satisfied_events = customer_session.list_events(
        metadata_filter=EventMetadataFilter(
            expression="customer_satisfaction = :sat",
            values={
                ":sat": StringValue("satisfied")
            }
        )
    )
    
    print(f"😊 Found {len(satisfied_events)} satisfied customer interaction(s)")
    
    print("\n💡 Support analytics use cases:")
    print("   - Track resolution rates by issue type")
    print("   - Measure average resolution time")
    print("   - Identify escalation patterns")
    print("   - Monitor customer satisfaction trends")
    print("   - Generate performance reports for support teams")
    
except Exception as e:
    logger.error(f"Error querying metadata: {e}")
    print(f"Note: Metadata filtering requires events with metadata tags")

#### Customer Support Tutorial completed! 🎉
Key takeaways:
- Memory hooks automatically manage customer context across support sessions using MemorySessionManager
- Multi-strategy memory captures orders, preferences, and facts from conversations
- Agents can provide personalized support based on customer history
- Tools can be integrated for order lookup and web search functionality
- Customer support becomes more efficient with persistent memory
- Session-based API eliminates repetitive parameter passing
- **Branching enables exploring alternative support resolution paths**
- **Metadata provides powerful support analytics and performance tracking**

## Clean Up

### Optional: Delete Memory Resource

In [ ]:
# Uncomment to delete the memory resource
# try:
#     client.delete_memory_and_wait(memory_id=memory_id)
#     print(f"✅ Deleted memory resource: {memory_id}")
# except Exception as e:
#     print(f"Error deleting memory: {e}")